In [1]:
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np

# 1. Load your CSV chunk data
csv_path = r"C:\Users\japje\Downloads\wiki_hybrid_chunks_300.csv"
df = pd.read_csv(csv_path)



# 2. Load sentence transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Encode text chunks to vectors
texts = df["chunk_text"].tolist()
print("Encoding documents...")

embeddings = model.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=True)
embedding_dim = embeddings.shape[1]  # e.g., 384 for MiniLM

# 4. Create FAISS index (flat = exact search, fast for small-medium datasets)
index = faiss.IndexFlatL2(embedding_dim)

# 5. Add embeddings to index
index.add(embeddings)

# 6. Store mapping of index to text
id_to_text = df[["chunk_id", "chunk_text"]].reset_index(drop=True)

# --------- RAG-Style Search Loop ---------
print("\n📚 FAISS Retriever is ready!")
print("Type your question below (or type 'exit' to quit):\n")

while True:
    query = input("Your question: ").strip().lower()
    if query == "exit":
        print("Exiting. Goodbye!")
        break

    # Embed the query
    query_vec = model.encode([query], convert_to_numpy=True)
    
    # Search top-k results
    k = 5
    distances, indices = index.search(query_vec, k)

    print("\n🔎 Top Results:\n")
    for idx in indices[0]:
        chunk = id_to_text.iloc[idx]
        print(f"Chunk ID: {chunk['chunk_id']}")
        print(f"Text: {chunk['chunk_text']}\n---")


Encoding documents...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]


📚 FAISS Retriever is ready!
Type your question below (or type 'exit' to quit):



Your question:  who is Marieve herington



🔎 Top Results:

Chunk ID: FullText_4
Text: com e marieve herington facebook marieve herington on twitter e marieve herington on instagram categories 1988 births living people actresses from los angeles actresses from oakville, ontario canadian child actresses canadian emigrants to the united states canadian film actresses canadian jazz singers canadian television actresses canadian video game actresses canadian voice actresses canadian women jazz singers franco ontarian people jazz musicians from california singers from los angeles university of toronto alumni 21st century canadian actresses 21st century canadian women singers
---
Chunk ID: FullText_2
Text: and members of the royal jelly orchestra, appearing on two more albums. she sang the theme songs for cbc television s sesame park, nelvana s pippi longstocking and tvontario s marigold s mathemagics. herington moved to los angeles in 2008. 2i the marieve herington band performs in southern california and toronto. their latest album

Your question:  who is bob brown



🔎 Top Results:

Chunk ID: FullText_1
Text: for other people with the same name, see rob brown. robert william brown born april 10, 1968 is a canadian former professional ice hockey right winger. rob brown he is best known for his time spent playing for the pittsburgh penguins from his debut in 1987 until 1990, and then again from 1997 until 2000. between and following these stints, brown shuffled between minor league teams in the international hockey league ihl and other nhl teams, including the hartford whalers, chicago blackhawks, dallas stars, and los angeles kings. playing career as a youth, he played in the 1981 quebec international pee wee hockey tournament with a minor ice hockey team from oshawa. brown was a prolific scorer at the junior level, averaging over two points per game brown in 2010 during his junior career. in particular, brown flourished in 1986 87 winning multiple born april 10, 1968 age 57 awards including most valuable player west , top scorer west , and the ina

Your question:  exit


Exiting. Goodbye!
